# Robust Detection of AI-Generated Images Under Real-World Transformations

**TikTok TechJam 2026 — Track 5**

Trains a ResNet18-based binary classifier (real vs. AI-generated) on the **CIFAKE** dataset, with training-time augmentation drawn from the same transform families the organizers evaluate robustness against (JPEG re-encode, Gaussian blur, resize-down/up, Gaussian noise, color jitter, center crop). Evaluates on 14 fixed conditions (clean + all org-specified severities), reports a robustness table, does error analysis, and runs an optional out-of-domain sanity check on the org-provided WildFake validation subset.

**Runs entirely on Kaggle**: enable GPU (Settings → Accelerator → GPU T4 x2 or P100), add the input dataset **`birdy654/cifake-real-and-ai-generated-synthetic-images`**, then Run All.

See `.bureau/contracts/direction_v1.md` in the project repo for the full design rationale.

In [ ]:
import os, io, json, random, time, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms as T
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected. On Kaggle, set Settings -> Accelerator -> GPU T4 x2 or P100.")

In [ ]:
# ---- Config ----
# Kaggle mounts a dataset at /kaggle/input/<dataset-slug>/..., but the exact nesting can
# vary — e.g. if attached via a notebook's output rather than as a plain dataset input, it
# can land under /kaggle/input/notebooks/<hash>/<dataset-slug>/... instead. Rather than
# guess, we (1) try a couple of known fast-path candidates, then (2) fall back to scanning
# under /kaggle/input for whichever folder actually contains train/{REAL,FAKE} and
# test/{REAL,FAKE}.
DATA_ROOT_FAST_CANDIDATES = [
    "/kaggle/input/cifake-real-and-ai-generated-synthetic-images",
    "/kaggle/input/notebooks/y5cy5c/cifake-real-and-ai-generated-synthetic-images",
]
DATA_ROOT_SEARCH_BASES = [
    "/kaggle/input/notebooks",
    "/kaggle/input",
]

def _has_cifake_layout(path):
    return (os.path.isdir(os.path.join(path, "train", "REAL")) and
            os.path.isdir(os.path.join(path, "train", "FAKE")) and
            os.path.isdir(os.path.join(path, "test", "REAL")) and
            os.path.isdir(os.path.join(path, "test", "FAKE")))

def find_cifake_root(fast_candidates, search_bases, max_depth=4):
    for p in fast_candidates:
        if _has_cifake_layout(p):
            return p
    for base in search_bases:
        if not os.path.isdir(base):
            continue
        stack = [(base, 0)]
        while stack:
            path, depth = stack.pop()
            if _has_cifake_layout(path):
                return path
            if depth < max_depth:
                try:
                    for entry in os.scandir(path):
                        if entry.is_dir():
                            stack.append((entry.path, depth + 1))
                except PermissionError:
                    pass
    return None

DATA_ROOT = find_cifake_root(DATA_ROOT_FAST_CANDIDATES, DATA_ROOT_SEARCH_BASES) or DATA_ROOT_FAST_CANDIDATES[0]

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR = os.path.join(DATA_ROOT, "test")

# Optional: org-provided out-of-domain validation subset (WildFake: COCO val2017 = real, DALL-E Advanced = fake).
# Not required to run the core pipeline; the OOD cell below skips gracefully if absent.
OOD_ROOT_CANDIDATES = [
    "/kaggle/input/wildfake-validation-subset",
    "/kaggle/input/wildfake",
]
OOD_ROOT = next((p for p in OOD_ROOT_CANDIDATES if os.path.isdir(p)), None)

IMG_SIZE = 128           # upscale target: makes org transform severities meaningful (native CIFAKE is 32x32)
BATCH_SIZE = 128
EPOCHS = 12
LR = 3e-4
WEIGHT_DECAY = 1e-4
VAL_FRACTION = 0.1       # carved out of the CIFAKE train split
AUG_PROB = 0.5           # fraction of training batches that receive one robustness-family augmentation
EARLY_STOP_PATIENCE = 3
EVAL_SAMPLES_PER_CONDITION = 3000   # cap per the direction contract, to fit a single Kaggle session
OOD_EVAL_SAMPLES = 500

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./working"
os.makedirs(OUT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(OUT_DIR, "resnet18_aigc_detector.pt")

print("DATA_ROOT:", DATA_ROOT, "exists:", os.path.isdir(DATA_ROOT))
print("OOD_ROOT:", OOD_ROOT)
print("OUT_DIR:", OUT_DIR)

## Transform library

One shared set of PIL-based transform functions is used for both **training-time augmentation** (randomized severities, to teach invariance) and **eval-time robustness conditions** (fixed severities, matching the organizer's spec exactly). Keeping them in one place guarantees the eval conditions are applied consistently and the training augmentation genuinely overlaps the same transform families without leaking the exact eval severities.

In [ ]:
# ---- Transform families (operate on PIL.Image, RGB) ----

def apply_jpeg(img: Image.Image, quality: int) -> Image.Image:
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

def apply_blur(img: Image.Image, sigma: float) -> Image.Image:
    radius = max(sigma, 0.1)
    return img.filter(ImageFilter.GaussianBlur(radius=radius))

def apply_resize_roundtrip(img: Image.Image, scale: float) -> Image.Image:
    w, h = img.size
    small = img.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.BILINEAR)
    return small.resize((w, h), Image.BILINEAR)

def apply_noise(img: Image.Image, sigma: float) -> Image.Image:
    arr = np.asarray(img).astype(np.float32) / 255.0
    noise = np.random.normal(0, sigma, arr.shape).astype(np.float32)
    noisy = np.clip(arr + noise, 0.0, 1.0)
    return Image.fromarray((noisy * 255).astype(np.uint8))

def apply_color_jitter(img: Image.Image, strength: float = 0.2) -> Image.Image:
    jitter = T.ColorJitter(brightness=strength, contrast=strength, saturation=strength)
    return jitter(img)

def apply_center_crop(img: Image.Image, keep_frac: float) -> Image.Image:
    w, h = img.size
    cw, ch = int(w * keep_frac), int(h * keep_frac)
    left, top = (w - cw) // 2, (h - ch) // 2
    cropped = img.crop((left, top, left + cw, top + ch))
    return cropped.resize((w, h), Image.BILINEAR)

# Fixed eval conditions per the organizer's spec (name -> callable applied to a PIL image)
EVAL_CONDITIONS = {
    "clean": lambda im: im,
    "jpeg_q90": lambda im: apply_jpeg(im, 90),
    "jpeg_q70": lambda im: apply_jpeg(im, 70),
    "jpeg_q50": lambda im: apply_jpeg(im, 50),
    "jpeg_q30": lambda im: apply_jpeg(im, 30),
    "blur_s0.5": lambda im: apply_blur(im, 0.5),
    "blur_s1.0": lambda im: apply_blur(im, 1.0),
    "blur_s2.0": lambda im: apply_blur(im, 2.0),
    "resize_0.5x": lambda im: apply_resize_roundtrip(im, 0.5),
    "resize_0.25x": lambda im: apply_resize_roundtrip(im, 0.25),
    "noise_s0.02": lambda im: apply_noise(im, 0.02),
    "noise_s0.05": lambda im: apply_noise(im, 0.05),
    "noise_s0.10": lambda im: apply_noise(im, 0.10),
    "color_jitter_20pct": lambda im: apply_color_jitter(im, 0.2),
    "center_crop_80pct": lambda im: apply_center_crop(im, 0.8),
}
assert len(EVAL_CONDITIONS) == 15  # clean + 14 transformed conditions

def random_training_augmentation(img: Image.Image) -> Image.Image:
    """Applies exactly one randomly-sampled robustness-family augmentation at a
    randomized severity (distinct range from the fixed eval severities above),
    to teach invariance without leaking the exact eval conditions."""
    family = random.choice(["jpeg", "blur", "resize", "noise", "jitter", "crop"])
    if family == "jpeg":
        return apply_jpeg(img, random.randint(40, 95))
    if family == "blur":
        return apply_blur(img, random.uniform(0.3, 2.5))
    if family == "resize":
        return apply_resize_roundtrip(img, random.uniform(0.4, 0.9))
    if family == "noise":
        return apply_noise(img, random.uniform(0.01, 0.12))
    if family == "jitter":
        return apply_color_jitter(img, random.uniform(0.1, 0.25))
    if family == "crop":
        return apply_center_crop(img, random.uniform(0.75, 0.95))
    return img

In [ ]:
# ---- Dataset ----

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def list_labeled_files(split_dir):
    """CIFAKE layout: <split_dir>/REAL/*.jpg and <split_dir>/FAKE/*.jpg. label: 1 = AI-generated (FAKE), 0 = real."""
    files = []
    for label_name, label in [("REAL", 0), ("FAKE", 1)]:
        folder = os.path.join(split_dir, label_name)
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            for fp in glob.glob(os.path.join(folder, ext)):
                files.append((fp, label))
    random.Random(SEED).shuffle(files)
    return files

class CifakeDataset(Dataset):
    def __init__(self, samples, img_size=IMG_SIZE, train=False, aug_prob=AUG_PROB):
        self.samples = samples
        self.img_size = img_size
        self.train = train
        self.aug_prob = aug_prob
        self.normalize = T.Compose([
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB").resize((self.img_size, self.img_size), Image.BICUBIC)
        if self.train and random.random() < self.aug_prob:
            img = random_training_augmentation(img)
        tensor = self.normalize(img)
        return tensor, torch.tensor(label, dtype=torch.float32), path

def build_dataloaders():
    train_all = list_labeled_files(TRAIN_DIR)
    n_val = int(len(train_all) * VAL_FRACTION)
    val_samples = train_all[:n_val]
    train_samples = train_all[n_val:]
    test_samples = list_labeled_files(TEST_DIR)

    print(f"train={len(train_samples)} val={len(val_samples)} test={len(test_samples)}")

    train_ds = CifakeDataset(train_samples, train=True)
    val_ds = CifakeDataset(val_samples, train=False)
    test_ds = CifakeDataset(test_samples, train=False)

    num_workers = 2 if DEVICE.type == "cuda" else 0
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_ds, test_samples

if os.path.isdir(TRAIN_DIR) and os.path.isdir(TEST_DIR):
    train_loader, val_loader, test_ds, test_samples = build_dataloaders()
else:
    train_loader = val_loader = test_ds = test_samples = None
    print(f"CIFAKE not found at {DATA_ROOT}. Add the Kaggle dataset "
          f"'birdy654/cifake-real-and-ai-generated-synthetic-images' as a notebook input, then re-run this cell.")

## Model

ResNet18 (ImageNet-pretrained), fully fine-tuned, single sigmoid logit output (`pred` = P(image is AI-generated)). ~11.7M parameters — far under the 2B-parameter limit, and fast enough to fully fine-tune within a single Kaggle GPU session.

In [ ]:
def build_model():
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 1)  # single logit; BCEWithLogitsLoss + sigmoid at inference
    return model.to(DEVICE)

model = build_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: ResNet18, {n_params:,} parameters (limit: 2,000,000,000)")
assert n_params < 2_000_000_000

## Training

AdamW + cosine LR decay, early stopping on validation loss (patience configured in the Config cell). Checkpoint is saved to `/kaggle/working` so re-running downstream cells doesn't require retraining.

In [ ]:
def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    criterion = nn.BCEWithLogitsLoss()
    total_loss, total_correct, total_n = 0.0, 0, 0
    for xb, yb, _ in loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train_mode):
            logits = model(xb).squeeze(1)
            loss = criterion(logits, yb)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * xb.size(0)
        preds = (torch.sigmoid(logits) > 0.5).float()
        total_correct += (preds == yb).sum().item()
        total_n += xb.size(0)
    return total_loss / total_n, total_correct / total_n

def train_model(model, train_loader, val_loader, epochs=EPOCHS):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_val_loss = float("inf")
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, optimizer=None)
        scheduler.step()
        dt = time.time() - t0
        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                         "val_loss": val_loss, "val_acc": val_acc, "seconds": dt})
        print(f"epoch {epoch:02d}/{epochs} | train_loss {train_loss:.4f} acc {train_acc:.4f} "
              f"| val_loss {val_loss:.4f} acc {val_acc:.4f} | {dt:.1f}s")

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save(model.state_dict(), CHECKPOINT_PATH)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOP_PATIENCE:
                print(f"Early stopping at epoch {epoch} (no val_loss improvement for {EARLY_STOP_PATIENCE} epochs).")
                break

    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
    return model, history

if train_loader is not None:
    model, history = train_model(model, train_loader, val_loader)
else:
    history = []
    print("Skipping training: dataloaders not built (CIFAKE input missing). Add the dataset and re-run from the Dataset cell.")

## Robustness evaluation

Evaluates the trained model on a fixed subsample of the CIFAKE test split under each of the 15 conditions (clean + 14 organizer-specified transforms), reporting accuracy, precision, recall, and AUC per condition. Sample size per condition is capped by `EVAL_SAMPLES_PER_CONDITION` (disclosed in the README, not hidden) to keep total eval time inside a single Kaggle session.

**Pre-registered success bar** (from the direction contract): mean accuracy across all 15 conditions ≥ 85%, and no single condition more than 15 points below the `clean` accuracy.

In [ ]:
class TransformedEvalDataset(Dataset):
    """Applies one fixed named transform (from EVAL_CONDITIONS) to each sample, for robustness eval."""
    def __init__(self, samples, transform_fn, img_size=IMG_SIZE):
        self.samples = samples
        self.transform_fn = transform_fn
        self.img_size = img_size
        self.normalize = T.Compose([T.ToTensor(), T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB").resize((self.img_size, self.img_size), Image.BICUBIC)
        img = self.transform_fn(img)
        return self.normalize(img), torch.tensor(label, dtype=torch.float32), path

@torch.no_grad()
def predict_probs(model, samples, transform_fn, batch_size=BATCH_SIZE):
    ds = TransformedEvalDataset(samples, transform_fn)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2 if DEVICE.type == "cuda" else 0)
    model.eval()
    all_probs, all_labels, all_paths = [], [], []
    for xb, yb, paths in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        logits = model(xb).squeeze(1)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(yb.numpy().tolist())
        all_paths.extend(paths)
    return np.array(all_probs), np.array(all_labels), all_paths

def run_robustness_eval(model, test_samples, n_per_condition=EVAL_SAMPLES_PER_CONDITION):
    rng = random.Random(SEED)
    eval_pool = test_samples.copy()
    rng.shuffle(eval_pool)
    eval_subset = eval_pool[:min(n_per_condition, len(eval_pool))]

    rows = []
    per_condition_raw = {}
    for name, fn in EVAL_CONDITIONS.items():
        probs, labels, paths = predict_probs(model, eval_subset, fn)
        preds = (probs > 0.5).astype(int)
        acc = accuracy_score(labels, preds)
        prec = precision_score(labels, preds, zero_division=0)
        rec = recall_score(labels, preds, zero_division=0)
        try:
            auc = roc_auc_score(labels, probs)
        except ValueError:
            auc = float("nan")  # only one class present in the sample
        rows.append({"condition": name, "n": len(eval_subset), "accuracy": acc,
                      "precision": prec, "recall": rec, "auc": auc})
        per_condition_raw[name] = (probs, labels, paths)
        print(f"{name:20s} n={len(eval_subset):5d} acc={acc:.4f} prec={prec:.4f} rec={rec:.4f} auc={auc:.4f}")

    return rows, per_condition_raw, eval_subset

if test_samples is not None:
    robustness_rows, per_condition_raw, eval_subset = run_robustness_eval(model, test_samples)
else:
    robustness_rows, per_condition_raw, eval_subset = [], {}, []
    print("Skipping robustness eval: no test set available.")

In [ ]:
import csv

def save_and_report_robustness(rows, out_path=os.path.join(OUT_DIR, "robustness_table.csv")):
    if not rows:
        print("No robustness rows to report.")
        return
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved robustness table -> {out_path}")

    clean_acc = next(r["accuracy"] for r in rows if r["condition"] == "clean")
    mean_acc = np.mean([r["accuracy"] for r in rows])
    worst = min(rows, key=lambda r: r["accuracy"])
    max_drop = (clean_acc - worst["accuracy"]) * 100

    print(f"\nclean accuracy      : {clean_acc:.4f}")
    print(f"mean accuracy (all)  : {mean_acc:.4f}")
    print(f"worst condition      : {worst['condition']} (acc {worst['accuracy']:.4f}, "
          f"-{max_drop:.1f}pt vs clean)")

    bar_pass = mean_acc >= 0.85 and max_drop <= 15.0
    print(f"\nDirection-contract success bar (mean acc >= 0.85 AND max drop <= 15pt): "
          f"{'PASS' if bar_pass else 'MISS'}")
    return mean_acc, max_drop, bar_pass

def plot_robustness(rows):
    if not rows:
        return
    names = [r["condition"] for r in rows]
    accs = [r["accuracy"] for r in rows]
    plt.figure(figsize=(11, 4))
    colors = ["tab:green" if n == "clean" else "tab:blue" for n in names]
    plt.bar(names, accs, color=colors)
    plt.axhline(0.85, color="red", linestyle="--", linewidth=1, label="85% success bar")
    plt.xticks(rotation=60, ha="right")
    plt.ylabel("Accuracy")
    plt.title("Robustness across transform conditions")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "robustness_plot.png"), dpi=150)
    plt.show()

save_and_report_robustness(robustness_rows)
plot_robustness(robustness_rows)

## Error analysis

Sample grids of misclassified images per condition, to inspect failure modes visually rather than
relying on aggregate numbers alone. We pick the two conditions with the largest accuracy drop from
`clean` (typically the harshest blur/resize settings) and show a handful of false positives (real
predicted as fake) and false negatives (fake predicted as real).

In [ ]:
def show_failure_grid(condition_name, per_condition_raw, max_examples=4):
    if condition_name not in per_condition_raw:
        print(f"No data for condition '{condition_name}'.")
        return
    probs, labels, paths = per_condition_raw[condition_name]
    preds = (probs > 0.5).astype(int)

    fp_idx = np.where((preds == 1) & (labels == 0))[0][:max_examples]  # real predicted fake
    fn_idx = np.where((preds == 0) & (labels == 1))[0][:max_examples]  # fake predicted real

    rows_idx = [("false positive (real -> pred fake)", fp_idx), ("false negative (fake -> pred real)", fn_idx)]
    n_cols = max_examples
    fig, axes = plt.subplots(2, n_cols, figsize=(3 * n_cols, 6.5))
    fig.suptitle(f"Failure examples under '{condition_name}'")

    for row, (row_title, idxs) in enumerate(rows_idx):
        for col in range(n_cols):
            ax = axes[row][col]
            ax.axis("off")
            if col < len(idxs):
                i = idxs[col]
                img = Image.open(paths[i]).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
                img = EVAL_CONDITIONS[condition_name](img)
                ax.imshow(img)
                ax.set_title(f"p(fake)={probs[i]:.2f}", fontsize=9)
            if col == 0:
                ax.text(-0.15, 0.5, row_title, transform=ax.transAxes, rotation=90,
                        va="center", ha="right", fontsize=9)
    plt.tight_layout()
    fname = os.path.join(OUT_DIR, f"failures_{condition_name}.png")
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f"Saved -> {fname}")

if robustness_rows:
    clean_acc = next(r["accuracy"] for r in robustness_rows if r["condition"] == "clean")
    non_clean = [r for r in robustness_rows if r["condition"] != "clean"]
    worst_two = sorted(non_clean, key=lambda r: r["accuracy"])[:2]
    print("Rendering failure grids for worst conditions:", [r["condition"] for r in worst_two])
    for r in worst_two:
        show_failure_grid(r["condition"], per_condition_raw)
else:
    print("Skipping error analysis: no robustness eval results available.")

## Out-of-domain sanity check (WildFake, non-headline)

CIFAKE is a narrow, low-resolution proxy (32x32 CIFAR-10 photos vs. Stable-Diffusion fakes). To be
transparent about the resulting domain-mismatch risk, we run the same model — with zero retraining —
against the org-provided **WildFake** validation subset (COCO val2017 = real, DALL-E Advanced = fake),
which is explicitly demonstration-only per the brief. This result is **not** part of the pre-registered
success bar; it exists purely to disclose how far the CIFAKE-trained model generalizes beyond its
training domain. Skips gracefully if the dataset isn't attached as a Kaggle input.

In [ ]:
def find_ood_samples(root, max_samples=OOD_EVAL_SAMPLES):
    """Best-effort loader for the WildFake validation subset. Tries a few common layouts:
    <root>/REAL|FAKE/*, <root>/real|fake/*, or <root>/coco*|dalle*/* (case-insensitive)."""
    if root is None:
        return []
    candidates = [
        ("REAL", "FAKE"), ("real", "fake"), ("Real", "Fake"),
    ]
    samples = []
    for real_name, fake_name in candidates:
        real_dir = os.path.join(root, real_name)
        fake_dir = os.path.join(root, fake_name)
        if os.path.isdir(real_dir) and os.path.isdir(fake_dir):
            for ext in ("*.jpg", "*.jpeg", "*.png"):
                samples += [(p, 0) for p in glob.glob(os.path.join(real_dir, ext))]
                samples += [(p, 1) for p in glob.glob(os.path.join(fake_dir, ext))]
            break
    if not samples:
        # fallback: any subfolder containing "coco" -> real, containing "dalle" -> fake
        for sub in glob.glob(os.path.join(root, "*")):
            if not os.path.isdir(sub):
                continue
            name = os.path.basename(sub).lower()
            label = 0 if "coco" in name else (1 if "dall" in name else None)
            if label is None:
                continue
            for ext in ("*.jpg", "*.jpeg", "*.png"):
                samples += [(p, label) for p in glob.glob(os.path.join(sub, ext))]

    random.Random(SEED).shuffle(samples)
    return samples[:max_samples]

ood_samples = find_ood_samples(OOD_ROOT)

if ood_samples and test_samples is not None:
    ood_probs, ood_labels, ood_paths = predict_probs(model, ood_samples, EVAL_CONDITIONS["clean"])
    ood_preds = (ood_probs > 0.5).astype(int)
    ood_acc = accuracy_score(ood_labels, ood_preds)
    ood_prec = precision_score(ood_labels, ood_preds, zero_division=0)
    ood_rec = recall_score(ood_labels, ood_preds, zero_division=0)
    try:
        ood_auc = roc_auc_score(ood_labels, ood_probs)
    except ValueError:
        ood_auc = float("nan")
    print(f"[OOD / non-headline] WildFake n={len(ood_samples)} "
          f"acc={ood_acc:.4f} prec={ood_prec:.4f} rec={ood_rec:.4f} auc={ood_auc:.4f}")
    print("This number reflects zero-shot transfer to a different generator/domain and is NOT "
          "part of the pre-registered success bar.")
else:
    print("Skipping OOD sanity check: WildFake dataset not attached as a Kaggle input "
          "(expected at one of:", OOD_ROOT_CANDIDATES, ") or model not trained.")

## Inference contract

Given an arbitrary directory of images, produce the deliverable JSON output:
`[{"image_path": ..., "pred": <float 0-1, probability the image is AI-generated>}]`. Threshold-free —
downstream consumers pick their own operating point.

In [ ]:
@torch.no_grad()
def predict_dir(image_dir, out_json_path=None, batch_size=BATCH_SIZE):
    """Runs the trained model over every image in `image_dir` (non-recursive) and writes/returns
    a JSON list of {"image_path": ..., "pred": float} — pred = P(image is AI-generated)."""
    paths = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        paths += glob.glob(os.path.join(image_dir, ext))
    paths = sorted(set(paths))
    if not paths:
        print(f"No images found in {image_dir}.")
        return []

    normalize = T.Compose([T.ToTensor(), T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])
    model.eval()
    results = []
    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i + batch_size]
        imgs = []
        for p in batch_paths:
            img = Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
            imgs.append(normalize(img))
        xb = torch.stack(imgs).to(DEVICE)
        logits = model(xb).squeeze(1)
        probs = torch.sigmoid(logits).cpu().numpy()
        for p, prob in zip(batch_paths, probs):
            results.append({"image_path": p, "pred": float(prob)})

    out_json_path = out_json_path or os.path.join(OUT_DIR, "predictions.json")
    with open(out_json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Wrote {len(results)} predictions -> {out_json_path}")
    return results

# Example: run inference over the CIFAKE test/FAKE folder as a smoke test (swap in any target directory).
if test_samples is not None:
    _example_dir = os.path.join(TEST_DIR, "FAKE")
    if os.path.isdir(_example_dir):
        _preds = predict_dir(_example_dir)
        print(_preds[:3])
else:
    print("predict_dir(image_dir) is ready to use once the model is trained.")

## Limitations (disclosed, not hidden)

- **Domain mismatch:** CIFAKE's real images are native 32x32 CIFAR-10 photos and its fakes come from
  one generator family (Stable Diffusion). Upscaling to 128px makes the org's transform severities
  meaningful, but it cannot manufacture detail that was never captured — the model may still learn
  resolution- or upscaling-artifact shortcuts rather than universal generative artifacts. The WildFake
  OOD check above is a directional signal, not proof of production-grade generalization.
- **Single generator family in training:** no exposure during training to diffusion/GAN variants beyond
  what CIFAKE contains (e.g. no Midjourney, no newer diffusion checkpoints). Real-world detection
  performance against generators absent from CIFAKE is unverified.
- **Eval sampling cap:** each of the 15 conditions is scored on a fixed subsample
  (`EVAL_SAMPLES_PER_CONDITION`, default 3,000) rather than the full test split, to fit a single Kaggle
  GPU session. This trades a small amount of statistical precision for guaranteed completion; the cap
  is applied uniformly and reported alongside every number.
- **OOD subset is small and demonstration-only:** per the original brief, WildFake here is a
  validation-only convenience subset, not a rigorously curated benchmark — treat its numbers as a
  sanity check, not a headline result.
- **Single architecture, no ensembling:** by design (see the direction contract) — ViT backbones,
  multi-model ensembles, and frequency-domain forensic features were explicitly deferred to keep this
  first edition shippable within one Kaggle session.
- **No adversarial robustness claims:** the transform battery covers common real-world post-processing
  (compression, blur, resize, noise, color, crop) but not adversarially-optimized perturbations aimed
  at fooling this specific model.